## 1. Import Libraries

In [1]:
import os
import re
import html
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Project Paths

In [35]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

INTERIM_DATA = os.path.join(
    DATA_PATH,
    "interim"
)

REPORT_PATH = os.path.join(
    PROJECT_PATH,
    "reports"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

## 3. Load Master Dataset

In [3]:
master_df = pd.read_parquet(

    os.path.join(
        INTERIM_DATA,
        "master_news_dataset.parquet"
    )

)

In [4]:
master_df.shape

(3215315, 15)

In [5]:
master_df.head()

,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,weekday,hour,headline_length,word_count,has_time
0,1,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 14:30:54+00:00,A,analyst,2020,6,5,Friday,14,39,7,True
1,2,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 14:45:20+00:00,A,analyst,2020,6,3,Wednesday,14,42,7,True
2,3,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 08:30:07+00:00,A,analyst,2020,5,26,Tuesday,8,29,5,True
3,4,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 16:45:06+00:00,A,analyst,2020,5,22,Friday,16,44,7,True
4,5,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 15:38:59+00:00,A,analyst,2020,5,22,Friday,15,87,14,True


In [6]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3215315 entries, 0 to 3215314
Data columns (total 15 columns):
 #   Column           Dtype              
---  ------           -----              
 0   news_id          int64              
 1   headline         object             
 2   url              object             
 3   publisher        category           
 4   published_date   datetime64[ns, UTC]
 5   ticker           category           
 6   source           category           
 7   year             int32              
 8   month            int32              
 9   day              int32              
 10  weekday          category           
 11  hour             int32              
 12  headline_length  int64              
 13  word_count       int64              
 14  has_time         bool               
dtypes: bool(1), category(4), datetime64[ns, UTC](1), int32(4), int64(3), object(2)
memory usage: 218.1+ MB


In [7]:
master_df.columns.tolist()

['news_id',
 'headline',
 'url',
 'publisher',
 'published_date',
 'ticker',
 'source',
 'year',
 'month',
 'day',
 'weekday',
 'hour',
 'headline_length',
 'word_count',
 'has_time']

## 4. Keep Original Headline

In [8]:
master_df["original_headline"] = master_df["headline"]

In [9]:
master_df[
    [
        "headline",
        "original_headline"
    ]
].head()

,headline,original_headline
0,Stocks That Hit 52-Week Highs On Friday,Stocks That Hit 52-Week Highs On Friday
1,Stocks That Hit 52-Week Highs On Wednesday,Stocks That Hit 52-Week Highs On Wednesday
2,71 Biggest Movers From Friday,71 Biggest Movers From Friday
3,46 Stocks Moving In Friday's Mid-Day Session,46 Stocks Moving In Friday's Mid-Day Session
4,B of A Securities Maintains Neutral on Agilent...,B of A Securities Maintains Neutral on Agilent...


## 5. Remove HTML Entities

In [10]:
master_df["headline"] = (

    master_df["original_headline"]

    .apply(html.unescape)

)

In [11]:
sample = master_df[
    master_df["headline"].str.contains("&", na=False)
]

sample[
    [
        "headline",
        "original_headline"
    ]
].head(10)

,headline,original_headline
141,Shares of several healthcare companies are tra...,Shares of several healthcare companies are tra...
143,Cowen & Co. Maintains Outperform on Agilent Te...,Cowen & Co. Maintains Outperform on Agilent Te...
198,Cowen & Co. Upgrades Agilent Technologies to O...,Cowen & Co. Upgrades Agilent Technologies to O...
216,"Bank Of America Out Positive On Agilent, Raise...","Bank Of America Out Positive On Agilent, Raise..."
223,The 10 Most Expensive Stocks In The S&P 500,The 10 Most Expensive Stocks In The S&P 500
229,Deutsche Bank Says Agilent Can Be The Next 'Co...,Deutsche Bank Says Agilent Can Be The Next 'Co...
353,Cowen & Company Upgrades Agilent Technologies ...,Cowen &amp; Company Upgrades Agilent Technolog...
403,"Earning & Economic Calendar for Monday May 18,...","Earning & Economic Calendar for Monday May 18,..."
420,S&P 500 Stocks With Most Active Options Today:...,S&P 500 Stocks With Most Active Options Today:...
436,Earning & Economic Calendar for Monday Novembe...,Earning & Economic Calendar for Monday Novembe...


## 9. Remove URLs

In [12]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")

In [13]:
def remove_urls(text):

    if pd.isna(text):

        return text

    return URL_PATTERN.sub("", text)

In [14]:
master_df["headline"] = (

    master_df["headline"]

    .apply(remove_urls)

)

In [15]:
master_df[
    master_df["headline"].str.contains(
        "http",
        na=False
    )
]

,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,weekday,hour,headline_length,word_count,has_time,original_headline


## 10. Normalize Whitespace

In [16]:
def normalize_spaces(text):

    if pd.isna(text):

        return text

    return " ".join(text.split())

In [17]:
master_df["headline"] = (

    master_df["headline"]

    .apply(normalize_spaces)

)

In [18]:
master_df[
    [
        "headline",
        "original_headline"
    ]
].sample(5, random_state=42)

,headline,original_headline
2451118,Investors Should Be More Interested In Herman ...,Investors Should Be More Interested In Herman ...
2190697,Venture Capital Deals Of The Week: Ant Financi...,Venture Capital Deals Of The Week: Ant Financi...
2225454,Here's Why You Should Retain IMAX in Your Port...,Here's Why You Should Retain IMAX in Your Port...
1470934,"Five Leaders, Two Bases As Pipelines Easily Ou...","Five Leaders, Two Bases As Pipelines Easily Ou..."
2683801,BlackBerry (BB) Gains But Lags Market: What Yo...,BlackBerry (BB) Gains But Lags Market: What Yo...


## 11. Standardize Quotes

In [19]:
QUOTE_MAP = {

    "“": '"',

    "”": '"',

    "‘": "'",

    "’": "'"

}

In [20]:
def normalize_quotes(text):

    if pd.isna(text):

        return text

    for old, new in QUOTE_MAP.items():

        text = text.replace(old, new)

    return text

In [21]:
master_df["headline"] = (

    master_df["headline"]

    .apply(normalize_quotes)

)

## 12. Remove Invisible Characters

In [22]:
def remove_control_characters(text):

    if pd.isna(text):

        return text

    text = text.replace("\n", " ")

    text = text.replace("\r", " ")

    text = text.replace("\t", " ")

    return text

In [23]:
master_df["headline"] = (

    master_df["headline"]

    .apply(remove_control_characters)

)

In [24]:
master_df["headline"] = (

    master_df["headline"]

    .apply(normalize_spaces)

)

## 13. Remove Empty Headlines

In [25]:
master_df["headline"] = (

    master_df["headline"]

    .astype(str)

    .str.strip()

)

In [26]:
before = len(master_df)

master_df = master_df[
    master_df["headline"] != ""
].copy()

after = len(master_df)

print(f"Removed {before-after:,} empty headlines")

Removed 19 empty headlines


## 14. Remove Duplicate Clean Headlines

In [27]:
duplicate_text = (

    master_df["headline"]

    .duplicated()

    .sum()

)

print("Duplicate Clean Headlines :", duplicate_text)

Duplicate Clean Headlines : 1565367


## 15. Financial Text Validation

In [28]:
sample = master_df.sample(
    10,
    random_state=42
)

sample[
    [
        "headline",
        "original_headline"
    ]
]

,headline,original_headline
66763,Antero Midstream shares are trading lower afte...,Antero Midstream shares are trading lower afte...
1666857,Cnooc's Nexen cutting 340 jobs in North Americ...,Cnooc's Nexen cutting 340 jobs in North Americ...
2344334,Valuation Dashboard: Energy And Materials - Up...,Valuation Dashboard: Energy And Materials - Up...
2050799,China Faces Tough Decisions Ahead,China Faces Tough Decisions Ahead
1549633,Dividend Challenger Highlights: Week Of May 26,Dividend Challenger Highlights: Week Of May 26
1066481,Regions Financial's Union Planters Preferred F...,Regions Financial's Union Planters Preferred F...
1983880,Google Confirms FCA Self-Driving Pact; Illumin...,Google Confirms FCA Self-Driving Pact; Illumin...
525532,Yen Weakens to 95/Dollar First Time Since Augu...,Yen Weakens to 95/Dollar First Time Since Augu...
377277,"Benzinga's Top Upgrades, Downgrades For Februa...","Benzinga's Top Upgrades, Downgrades For Februa..."
1349844,Benzinga's Top Downgrades,Benzinga's Top Downgrades


## 16. Character Statistics

In [29]:
master_df["clean_character_count"] = (

    master_df["headline"]

    .str.len()

)

## 17. Word Statistics

In [30]:
master_df["clean_word_count"] = (

    master_df["headline"]

    .str.split()

    .str.len()

)

## 18. Compare Original vs Clean

In [31]:
comparison = pd.DataFrame({

    "Original Characters":

        master_df["headline_length"],

    "Clean Characters":

        master_df["clean_character_count"],

    "Original Words":

        master_df["word_count"],

    "Clean Words":

        master_df["clean_word_count"]

})

comparison.describe()

,Original Characters,Clean Characters,Original Words,Clean Words
count,3.215296e+06,3.215296e+06,3.215296e+06,3.215296e+06
mean,6.429502e+01,6.428898e+01,1.016972e+01,1.016966e+01
std,3.180831e+01,3.180142e+01,4.949923e+00,4.949814e+00
min,2.000000e+00,2.000000e+00,1.000000e+00,1.000000e+00
25%,4.500000e+01,4.500000e+01,7.000000e+00,7.000000e+00
50%,5.800000e+01,5.800000e+01,9.000000e+00,9.000000e+00
75%,7.600000e+01,7.600000e+01,1.200000e+01,1.200000e+01
max,5.120000e+02,5.120000e+02,7.700000e+01,7.700000e+01


## 19. Quality Check

In [32]:
quality_report = pd.DataFrame({

    "Metric":[

        "Rows",

        "Original Headlines",

        "Clean Headlines",

        "Missing Headlines",

        "Missing Clean Headlines"

    ],

    "Value":[

        len(master_df),

        master_df["original_headline"].notna().sum(),

        master_df["headline"].notna().sum(),

        master_df["original_headline"].isna().sum(),

        master_df["headline"].isna().sum()

    ]

})

quality_report

,Metric,Value
0,Rows,3215296
1,Original Headlines,3215296
2,Clean Headlines,3215296
3,Missing Headlines,0
4,Missing Clean Headlines,0


## 20. Save Processed Dataset

In [36]:
master_df.to_parquet(

    os.path.join(

        PROCESSED_PATH,

        "financial_news_clean.parquet"

    ),

    index=False

)

## 21. Verify Saved File

In [37]:
test_df = pd.read_parquet(

    os.path.join(

        PROCESSED_PATH,

        "financial_news_clean.parquet"

    )

)

print(test_df.shape)

(3215296, 18)
